In [1]:
import random
random.seed(2026)

import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(">>> Using Apple Silicon (MPS) for acceleration.")
else:
    device = torch.device("cpu")
    print(">>> Using CPU.")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_path = './data'
is_downloaded = os.path.exists(os.path.join(data_path, 'oxford-iiit-pet'))

train_set = torchvision.datasets.OxfordIIITPet(
    root=data_path, split='trainval', target_types='category', 
    download=not is_downloaded, transform=transform
)

test_set = torchvision.datasets.OxfordIIITPet(
    root=data_path, split='test', target_types='category', 
    download=not is_downloaded, transform=transform
)

>>> Using Apple Silicon (MPS) for acceleration.


In [2]:
def train(dataloader, model, loss_fn, optimizer):
    # Get total dataset size
    size = len(dataloader.dataset)
    # Set model to training mode
    model.train()
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error (Forward pass)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation (Backward pass)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Print progress every 100 batches
        if batch % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    # Set model to evaluation mode
    model.eval()
    test_loss, correct = 0, 0
    
    # Disable gradient calculation for memory efficiency
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            # Accumulate loss and accuracy
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            
    # Calculate average loss and accuracy percentage
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [3]:
# 1. Define image transformations for Oxford Pet dataset
# Since these are real photos, we resize them to 224x224 and normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Download and load the training data
data_path = './data'
is_downloaded = os.path.exists(os.path.join(data_path, 'oxford-iiit-pet'))

training_data = torchvision.datasets.OxfordIIITPet(
    root=data_path, split='trainval', target_types='category', 
    download=not is_downloaded, transform=transform
)

test_data = torchvision.datasets.OxfordIIITPet(
    root=data_path, split='test', target_types='category', 
    download=not is_downloaded, transform=transform
)

batch_size = 32

train_dataloader = DataLoader(training_data, batch_size=batch_size, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=batch_size)


for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break



Shape of X [N, C, H, W]: torch.Size([32, 3, 224, 224])
Shape of y: torch.Size([32]) torch.int64


### Baseline

In [4]:
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
import os


# Define the Neural Network Architecture
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # Input layer: 224 (width) * 224 (height) * 3 (RGB channels) = 150,528
            nn.Linear(224 * 224 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            # Output layer: 37 classes for the Oxford Pet breeds
            nn.Linear(512, 37)
        )

    def forward(self, x):
        # Flatten the input image from (3, 224, 224) to a 1D vector
        x = self.flatten(x)
        # Pass the data through the linear layers
        logits = self.linear_relu_stack(x)
        return logits

# 4. Initialize model and move to device (GPU or CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = NeuralNetwork().to(device)

# Print the model structure
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=150528, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=37, bias=True)
  )
)


In [14]:
loss_fn = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # Adam 적용

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 3.411728  [   32/ 3680]
loss: 3.485141  [  352/ 3680]
loss: 3.604731  [  672/ 3680]
loss: 3.527833  [  992/ 3680]
loss: 3.593660  [ 1312/ 3680]
loss: 3.652204  [ 1632/ 3680]
loss: 3.572223  [ 1952/ 3680]
loss: 3.568799  [ 2272/ 3680]
loss: 3.584713  [ 2592/ 3680]
loss: 3.498704  [ 2912/ 3680]
loss: 3.794793  [ 3232/ 3680]
loss: 3.642775  [ 3552/ 3680]
Test Error: 
 Accuracy: 2.8%, Avg loss: 3.633712 

Epoch 2
-------------------------------
loss: 3.283803  [   32/ 3680]
loss: 3.493515  [  352/ 3680]
loss: 3.589367  [  672/ 3680]
loss: 3.522258  [  992/ 3680]
loss: 3.565413  [ 1312/ 3680]
loss: 3.570782  [ 1632/ 3680]
loss: 3.567949  [ 1952/ 3680]
loss: 3.556173  [ 2272/ 3680]
loss: 3.505019  [ 2592/ 3680]
loss: 3.743562  [ 2912/ 3680]
loss: 3.555039  [ 3232/ 3680]
loss: 3.606473  [ 3552/ 3680]
Test Error: 
 Accuracy: 2.9%, Avg loss: 3.647810 

Epoch 3
-------------------------------
loss: 3.149611  [   32/ 3680]
loss: 3.462790  [  352/ 3680

overfitting! 
Training Loss: $3.41 \to 3.11 \to 2.89$ (점점 내려감)Test Avg Loss: $3.63 \to 3.69 \to 3.88$ (점점 올라감!)

Epoch 1
-------------------------------
loss: 3.715918  [   32/ 3680]
loss: 3.625746  [ 3232/ 3680]
Test Error: 
 Accuracy: 6.9%, Avg loss: 3.506458 

Epoch 2
-------------------------------
loss: 3.814375  [   32/ 3680]
loss: 3.530391  [ 3232/ 3680]
Test Error: 
 Accuracy: 7.8%, Avg loss: 3.450248 

Epoch 3
-------------------------------
loss: 3.755675  [   32/ 3680]
loss: 3.462285  [ 3232/ 3680]
Test Error: 
 Accuracy: 7.9%, Avg loss: 3.418086 

Epoch 4
-------------------------------
loss: 3.619173  [   32/ 3680]
loss: 3.382589  [ 3232/ 3680]
Test Error: 
 Accuracy: 8.3%, Avg loss: 3.395835 

Epoch 5
-------------------------------
loss: 3.472992  [   32/ 3680]
loss: 3.301038  [ 3232/ 3680]
Test Error: 
 Accuracy: 8.7%, Avg loss: 3.380744 

Epoch 6
-------------------------------
loss: 3.312343  [   32/ 3680]
loss: 3.210962  [ 3232/ 3680]
Test Error: 
 Accuracy: 9.2%, Avg loss: 3.370891 

Epoch 7
-------------------------------
loss: 3.174283  [   32/ 3680]
loss: 3.123146  [ 3232/ 

### VGG 16

In [5]:
import torch
import torch.nn as nn

def conv_bn_relu(in_channels, out_channels):
    """
    Standard VGG Helper: 3x3 Conv -> Batch Normalization -> ReLU.
    VGG always uses kernel_size=3, stride=1, and padding=1 to keep dimensions the same.
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

class VGGBlock(nn.Module):
    """
    A VGG Block consists of multiple 3x3 convolutions followed by one MaxPool.
    - num_convs: Number of convolutional layers in this block.
    """
    def __init__(self, in_channels, out_channels, num_convs):
        super(VGGBlock, self).__init__()
        layers = []
        for i in range(num_convs):
            # The first conv changes the channel depth; others keep it the same.
            layers.append(conv_bn_relu(in_channels if i == 0 else out_channels, out_channels))
        
        # Max Pooling halves the height and width (e.g., 224x224 -> 112x112)
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class VGG16Custom(nn.Module):
    """
    VGG-16 Architecture:
    - 5 VGG Blocks (Total 13 Conv layers)
    - 3 Fully Connected layers (Total 3 Linear layers)
    - Total = 16 weight layers.
    """
    def __init__(self, num_classes=37):
        super(VGG16Custom, self).__init__()
        
        # ----- Feature Extraction (Convolutional Layers) -----
        self.features = nn.Sequential(
            VGGBlock(3, 64, 2),    # Block 1: 2 convs, 64 filters
            VGGBlock(64, 128, 2),  # Block 2: 2 convs, 128 filters
            VGGBlock(128, 256, 3), # Block 3: 3 convs, 256 filters
            VGGBlock(256, 512, 3), # Block 4: 3 convs, 512 filters
            VGGBlock(512, 512, 3)  # Block 5: 3 convs, 512 filters
        )

        # After 5 MaxPools (2^5 = 32), a 224x224 image becomes 7x7.
        # Channels are 512. Total = 512 * 7 * 7 = 25088.
        
        # ----- Classifier (Fully Connected Layers) -----
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5), # Regularization to prevent overfitting
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes) # Final output for 37 breeds
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate the custom model
model_vgg = VGG16Custom(num_classes=37).to(device)
print(model_vgg)



VGG16Custom(
  (features): Sequential(
    (0): VGGBlock(
      (block): Sequential(
        (0): Sequential(
          (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): VGGBlock(
      (block): Sequential(
        (0): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
     

In [ ]:
loss_fn = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model_vgg.parameters(),lr=1e-4)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_vgg, loss_fn, optimizer)
    test(test_dataloader, model_vgg, loss_fn)

print("Training Complete!")

Epoch 1
-------------------------------
loss: 3.573979  [   32/ 3680]
loss: 4.297650  [  352/ 3680]
loss: 3.961950  [  672/ 3680]
loss: 3.923028  [  992/ 3680]
loss: 3.736585  [ 1312/ 3680]
loss: 3.734648  [ 1632/ 3680]
loss: 3.534763  [ 1952/ 3680]
loss: 3.593003  [ 2272/ 3680]
loss: 3.603108  [ 2592/ 3680]
loss: 3.843229  [ 2912/ 3680]
loss: 3.714571  [ 3232/ 3680]
loss: 3.708423  [ 3552/ 3680]
Test Error: 
 Accuracy: 2.8%, Avg loss: 3.622117 

Epoch 2
-------------------------------
loss: 3.583422  [   32/ 3680]
loss: 3.572082  [  352/ 3680]
loss: 3.546336  [  672/ 3680]
loss: 3.835047  [  992/ 3680]
loss: 3.717549  [ 1312/ 3680]
loss: 3.658598  [ 1632/ 3680]
loss: 3.586785  [ 1952/ 3680]
loss: 3.599878  [ 2272/ 3680]
loss: 3.316250  [ 2592/ 3680]
loss: 3.807676  [ 2912/ 3680]
loss: 3.828327  [ 3232/ 3680]
loss: 3.650889  [ 3552/ 3680]
Test Error: 
 Accuracy: 2.7%, Avg loss: 3.612025 

Epoch 3
-------------------------------
loss: 3.599136  [   32/ 3680]
loss: 3.606360  [  352/ 3680

### ResNet34

In [15]:
import torch
import torch.nn as nn

# --- 1. Helper Function ---
def conv_bn_relu(in_channels, out_channels, kernel_size=3, stride=1, padding=1, apply_relu=True):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
        nn.BatchNorm2d(out_channels)
    ]
    if apply_relu:
        layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

# --- 2. Basic Block ) ---
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, is_plain=False):
        super(BasicBlock, self).__init__()
        self.is_plain = is_plain
        
        self.conv1 = conv_bn_relu(in_channels, out_channels, stride=stride)
        self.conv2 = conv_bn_relu(out_channels, out_channels, apply_relu=False)
        
        self.shortcut = nn.Sequential()
        if not is_plain:
            if stride != 1 or in_channels != out_channels:
                self.shortcut = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                    nn.BatchNorm2d(out_channels)
                )
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.conv2(out)

        if not self.is_plain:
            out += self.shortcut(identity)
            
        return self.relu(out)

# --- 3. Full ResNet-34 Class ---
class ResNet34Custom(nn.Module):
    def __init__(self, num_classes=37):
        super(ResNet34Custom, self).__init__()
        self.in_channels = 64

        # Initial Stem: 7x7 Conv -> BN -> ReLU -> MaxPool
        # This reduces a 224x224 image to 56x56
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet-34 Layers (Blocks: 3, 4, 6, 3)
        self.layer1 = self._make_layer(64,  3, stride=1) # Stage 1: 3 blocks (64 channels)
        self.layer2 = self._make_layer(128, 4, stride=2) # Stage 2: 4 blocks (128 channels)
        self.layer3 = self._make_layer(256, 6, stride=2) # Stage 3: 6 blocks (256 channels)
        self.layer4 = self._make_layer(512, 3, stride=2) # Stage 4: 3 blocks (512 channels)

        # Final Layers: Global Average Pool -> Fully Connected
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        layers = []
        # The first block of a layer handles the stride and channel expansion
        layers.append(BasicBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        
        # The remaining blocks keep the dimensions the same
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels))
            
        return nn.Sequential(*layers)

    def forward(self, x):
        # Input stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # ResNet stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Classification head
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Instantiate the custom ResNet-34
model_resnet = ResNet34Custom(num_classes=37).to(device)
print(model_resnet)



ResNet34Custom(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (shortcut): Sequential()
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1

In [ ]:
loss_fn = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model_resnet.parameters(),lr=1e-4) 


epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_resnet, loss_fn, optimizer) 
    test(test_dataloader, model_resnet, loss_fn)

print("Training Complete!")

### To make plots

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0 
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def test(dataloader, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            
    avg_loss = test_loss / len(dataloader)
    accuracy = (correct / len(dataloader.dataset)) * 100
    return avg_loss, accuracy

In [18]:
loss_fn = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model_resnet.parameters(),lr=1e-4) 


for t in range(epochs):
    print(f"Epoch {t+1}")
    train_loss = train(train_dataloader, model_resnet, loss_fn, optimizer)
    test_loss, test_acc = test(test_dataloader, model_resnet, loss_fn)
    
    # 리스트에 저장
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    print(f"Test Acc: {test_acc:>0.1f}%, Train Loss: {train_loss:>8f}")

Epoch 1


: 

Epoch 1
-------------------------------
loss: 4.417038  [   32/ 3680]
loss: 3.710548  [  352/ 3680]
loss: 4.548905  [  672/ 3680]
loss: 5.135700  [  992/ 3680]
loss: 4.075010  [ 1312/ 3680]
loss: 5.101553  [ 1632/ 3680]
loss: 3.598907  [ 1952/ 3680]
loss: 4.151203  [ 2272/ 3680]
loss: 3.957324  [ 2592/ 3680]
loss: 3.975146  [ 2912/ 3680]
loss: 4.760095  [ 3232/ 3680]
loss: 4.660428  [ 3552/ 3680]


KeyboardInterrupt: 